<a href="https://colab.research.google.com/github/carolineb3/Earnings-Call-NLP-Pipeline/blob/main/notebooks/01_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Data Ingestion

This notebook collects the raw inputs for the earnings call analysis. It loads S&P 500 earnings call transcripts from HuggingFace, pulls post-earnings stock price data from yfinance, and saves the transcript and return data needed for the rest of the pipeline.

The transcript data comes from `kurry/sp500_earnings_transcripts` and is filtered to FY2021–FY2024. For each available company and earnings date, the notebook also pulls stock price data and SPY benchmark returns so the project can calculate a short-window market reaction.

## What this notebook does

- Loads available FY2021–FY2024 earnings call transcripts.
- Identifies all available tickers in the selected year range.
- Pulls sector information once and caches it in `sector_map.csv`.
- Splits each transcript into prepared remarks and Q&A sections using speaker turns, with a keyword fallback when the structured split does not work.
- Pulls post-earnings stock prices from yfinance.
- Calculates the company’s short-window return and market-adjusted return when SPY benchmark data is available.
- Saves each successfully matched transcript as a JSON file.

## Outputs

This notebook creates three main outputs:

- `Transcripts_JSON/` — one JSON file per saved transcript
- `stock_prices.csv` — stock return, SPY return, and market-adjusted return data
- `sector_map.csv` — cached ticker-to-sector lookup table

## Validation checks

At the end, the notebook checks that:

- `stock_prices.csv` exists
- `Transcripts_JSON/` exists
- JSON transcript files were successfully created

In [ ]:
# =============================================================================
# STEP 1 — Data Ingestion
# Load earnings call transcripts, pull post-earnings price data, and save
# transcript JSONs plus return data for the downstream Spark pipeline.
# =============================================================================

!pip install pdfplumber yfinance datasets pyspark

from google.colab import drive
drive.mount('/content/drive')

import json
import os
import time
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
from datasets import load_dataset

BASE_DIR        = "/content/drive/MyDrive/MIS_584_Project"
TRANSCRIPTS_DIR = f"{BASE_DIR}/Transcripts_JSON"
PRICE_OUTPUT    = f"{BASE_DIR}/stock_prices.csv"

os.makedirs(TRANSCRIPTS_DIR, exist_ok=True)

START_YEAR    = 2021
END_YEAR      = 2024
RETURN_WINDOW = 3

print("Loading HuggingFace dataset...")
ds     = load_dataset("kurry/sp500_earnings_transcripts", split="train")
df_all = ds.to_pandas()

# Build the ticker list from the transcript records in the selected year range
ALL_TICKERS = sorted(
    df_all[df_all['year'].between(START_YEAR, END_YEAR)]['symbol'].unique().tolist()
)
print(f"Total tickers available in {START_YEAR}–{END_YEAR}: {len(ALL_TICKERS)}")

# Pull sector information once and cache it for later notebooks
SECTOR_CACHE = f"{BASE_DIR}/sector_map.csv"

if os.path.exists(SECTOR_CACHE):
    print("Loading sector map from cache...")
    sector_df  = pd.read_csv(SECTOR_CACHE)
    sector_map = dict(zip(sector_df['ticker'], sector_df['sector']))
else:
    print("Pulling sector info from yfinance...")
    sector_map = {}
    for ticker in ALL_TICKERS:
        try:
            info = yf.Ticker(ticker).info
            sector_map[ticker] = info.get('sector', 'Unknown')
        except Exception:
            sector_map[ticker] = 'Unknown'
        time.sleep(0.15)
    pd.DataFrame(
        list(sector_map.items()), columns=['ticker', 'sector']
    ).to_csv(SECTOR_CACHE, index=False)
    print(f"Sector map saved to {SECTOR_CACHE}")

print(f"Sectors found: {sorted(set(sector_map.values()))}")

print(f"Target: {len(ALL_TICKERS)} companies × "
      f"{END_YEAR - START_YEAR + 1} years × 4 quarters "
      f"= ~{len(ALL_TICKERS) * (END_YEAR - START_YEAR + 1) * 4} possible transcripts")

print(f"Full dataset: {len(df_all):,} transcripts")
print(f"Companies:    {df_all['symbol'].nunique():,}")
print(f"Date range:   {df_all['year'].min()} – {df_all['year'].max()}")
print(f"\nColumns: {df_all.columns.tolist()}")
print(df_all[['symbol', 'company_name', 'quarter', 'year', 'date']].head(3))

sample = df_all[df_all['symbol'] == 'KO'].iloc[0]
print(f"\nSample: {sample['symbol']} Q{sample['quarter']} {sample['year']}")
print(f"Number of turns: {len(sample['structured_content'])}")


def split_transcript_sections(structured_content, full_text):
    """Split transcript into prepared remarks and Q&A using speaker turns.
    Falls back to keyword search if structured content fails."""
    try:
        n_turns = len(structured_content) if structured_content is not None else 0
    except TypeError:
        n_turns = 0

    if n_turns == 0:
        return _keyword_split(full_text)

    qa_open_phrases = [
        'your first question', 'first question comes from',
        'first question is from', 'our first question comes from',
        'our first question is from', 'take your first question',
        'ready to take q&a', 'ready to take questions',
        'now open for questions', 'open for questions',
        'open the floor for questions', 'we will now begin the question',
        'now begin the question', 'we will now take questions',
        'begin the q&a', 'start the q&a',
    ]

    qa_close_phrases = [
        'concludes our question', 'concludes the question',
        'no further questions', 'that was our last question',
        'end of our q&a', 'end of the q&a', 'this concludes',
        'thank you for participating', 'thank you for joining',
        'conference has now concluded', 'turn the conference over',
        'turn the call back to management', 'turn the call over to management',
    ]

    qa_start_idx = None

    for i, turn in enumerate(structured_content):
        speaker = (turn.get('speaker') or '').lower().strip()
        text    = (turn.get('text')    or '').lower().strip()

        if 'operator' not in speaker or i == 0:
            continue
        if any(pat in text for pat in qa_close_phrases):
            continue
        if any(pat in text for pat in qa_open_phrases):
            qa_start_idx = i
            break

        speakers_so_far = {
            (structured_content[j].get('speaker') or '').lower().strip()
            for j in range(i)
            if (structured_content[j].get('speaker') or '').strip()
        }
        speakers_so_far.add('operator')

        if i + 1 < n_turns:
            next_speaker = (
                structured_content[i + 1].get('speaker') or ''
            ).lower().strip()
            if (next_speaker and
                next_speaker not in speakers_so_far and
                'operator' not in next_speaker):
                qa_start_idx = i
                break

    if qa_start_idx is None:
        return _keyword_split(full_text)

    prep_turns = structured_content[:qa_start_idx]
    qa_turns   = structured_content[qa_start_idx:]

    prepared_remarks = ' '.join(
        (t.get('text') or '') for t in prep_turns
    ).strip()
    qa_section = ' '.join(
        (t.get('text') or '') for t in qa_turns
    ).strip()

    if len(prepared_remarks.split()) < 50 or len(qa_section.split()) < 50:
        return _keyword_split(full_text)

    return prepared_remarks, qa_section


def _keyword_split(full_text):
    """Fallback: search raw text for Q&A section markers."""
    if not full_text:
        return '', ''

    markers = [
        'your first question comes from', 'your first question is from',
        'our first question comes from', 'our first question is from',
        'take your first question', 'ready to take q&a',
        'ready to take questions', 'now open for questions',
        'open for questions', 'open the floor for questions',
        'we will now take questions',
        'we will now begin the question-and-answer',
        'we will now begin the question and answer',
    ]

    text_lower = full_text.lower()
    best_pos   = -1

    for marker in markers:
        pos = text_lower.find(marker)
        if pos != -1 and pos > len(full_text) * 0.10:
            best_pos = pos
            break

    if best_pos == -1:
        return full_text.strip(), ''

    return full_text[:best_pos].strip(), full_text[best_pos:].strip()


# ── Filter to available tickers in selected year range ──────────────────────
df_cs = df_all[
    (df_all['symbol'].isin(ALL_TICKERS)) &
    (df_all['year'].between(START_YEAR, END_YEAR))
].copy().sort_values(['symbol', 'year', 'quarter']).reset_index(drop=True)

print(f"Total transcripts to process: {len(df_cs)}")
print(f"Unique companies: {df_cs['symbol'].nunique()}")

# ── Pull 3-day stock returns via yfinance ────────────────────────────────────
print("\nPulling stock prices from yfinance...")
price_records = []
failed        = []

df_cs['earnings_date'] = pd.to_datetime(df_cs['date']).dt.strftime('%Y-%m-%d')
pairs = df_cs[['symbol', 'earnings_date']].drop_duplicates()

for _, row in pairs.iterrows():
    ticker        = row['symbol']
    earnings_date = row['earnings_date']

    try:
        stock = yf.Ticker(ticker)
        end   = (
            datetime.strptime(earnings_date, '%Y-%m-%d') + timedelta(days=10)
        ).strftime('%Y-%m-%d')

        hist = stock.history(start=earnings_date, end=end)

        if hist.empty or len(hist) < 2:
            print(f"  ⚠ {ticker} {earnings_date}: not enough price data")
            failed.append((ticker, earnings_date))
            continue

        price_on_date  = float(hist['Close'].iloc[0])
        days_available = min(RETURN_WINDOW, len(hist) - 1)
        price_3d_later = float(hist['Close'].iloc[days_available])
        pct_return     = (price_3d_later - price_on_date) / price_on_date * 100

        # Pull SPY return over same window to market-adjust
        try:
            spy_hist   = yf.Ticker('SPY').history(start=earnings_date, end=end)
            spy_return = (
                float(spy_hist['Close'].iloc[days_available]) -
                float(spy_hist['Close'].iloc[0])
            ) / float(spy_hist['Close'].iloc[0]) * 100
            market_adj = round(pct_return - spy_return, 4)
        except Exception:
            spy_return = 0.0
            market_adj = round(pct_return, 4)

        price_records.append({
            'ticker':               ticker,
            'earnings_date':        earnings_date,
            'price_on_date':        round(price_on_date,  2),
            'price_3d_later':       round(price_3d_later, 2),
            'pct_return_3d':        round(pct_return,     4),
            'market_adj_return_3d': market_adj,
            'spy_return_3d':        round(spy_return,     4),
            'volume_on_date':       int(hist['Volume'].iloc[0]),
            'window_days_used':     days_available,
        })
        print(f"  ✓ {ticker} {earnings_date}: "
              f"${price_on_date:.2f} → ${price_3d_later:.2f} "
              f"({pct_return:+.2f}%, adj: {market_adj:+.2f}%)")

    except Exception as e:
        print(f"  ✗ {ticker} {earnings_date}: {e}")
        failed.append((ticker, earnings_date))

    time.sleep(0.5)

prices_df = pd.DataFrame(price_records)
prices_df.to_csv(PRICE_OUTPUT, index=False)
print(f"\nSaved {len(prices_df)} price records | Failed: {len(failed)}")

# ── Save matched transcripts as JSON files ───────────────────────────────────
print(f"\nParsing and saving {len(df_cs)} transcripts...")
saved   = 0
skipped = 0

for _, row in df_cs.iterrows():
    ticker        = row['symbol']
    quarter       = row['quarter']
    year          = row['year']
    earnings_date = row['earnings_date']
    quarter_label = f"Q{quarter}_FY{year}"

    price_match = prices_df[
        (prices_df['ticker']        == ticker) &
        (prices_df['earnings_date'] == earnings_date)
    ]
    if price_match.empty:
        print(f"  ⚠ Skipping {ticker} {quarter_label} — no price data")
        skipped += 1
        continue

    full_text                    = row['content'] or ''
    prepared_remarks, qa_section = split_transcript_sections(
        row['structured_content'], full_text
    )

    record = {
        'ticker':           ticker,
        'company_name':     row.get('company_name', ticker),
        'sector':           sector_map.get(ticker, 'Unknown'),
        'quarter':          quarter_label,
        'year':             int(year),
        'earnings_date':    earnings_date,
        'source':           'HuggingFace/kurry/sp500_earnings_transcripts',
        'full_text':        full_text,
        'prepared_remarks': prepared_remarks,
        'qa_section':       qa_section,
        'word_count':       len(full_text.split()),
        'prepared_words':   len(prepared_remarks.split()),
        'qa_words':         len(qa_section.split()),
    }

    filepath = os.path.join(TRANSCRIPTS_DIR, f"{ticker}_{quarter_label}.json")
    with open(filepath, 'w') as f:
        json.dump(record, f, indent=2)

    saved += 1

print(f"\nSaved: {saved} | Skipped: {skipped}")
print(f"\n{'='*60}\n  STEP 1 COMPLETE\n{'='*60}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 93.3 MB/s eta 0:00:00
Mounted at /content/drive
Loading HuggingFace dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

parquet_files/part-0.parquet:   0%|          | 0.00/1.82G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33362 [00:00<?, ? examples/s]

...

  ⚠ WBA 2021-01-07: not enough price data
  ⚠ WBA 2021-03-31: not enough price data
  ⚠ WBA 2021-07-01: not enough price data
  ⚠ WBA 2021-10-15: not enough price data
  ⚠ WBA 2022-01-06: not enough price data
  ⚠ WBA 2022-03-31: not enough price data
  ⚠ WBA 2022-06-30: not enough price data
  ⚠ WBA 2022-10-13: not enough price data
  ⚠ WBA 2023-01-05: not enough price data
  ⚠ WBA 2023-03-28: not enough price data
  ⚠ WBA 2023-06-27: not enough price data
  ⚠ WBA 2023-10-12: not enough price data
  ⚠ WBA 2024-01-04: not enough price data
  ⚠ WBA 2024-03-28: not enough price data
  ⚠ WBA 2024-06-27: not enough price data
  ⚠ WBA 2024-10-15: not enough price data
  ✓ WBD 2021-04-28: $37.49 → $36.12 (-3.65%, adj: -3.85%)
  ✓ WBD 2021-08-03: $27.87 → $29.04 (+4.20%, adj: +3.89%)
  ✓ WBD 2021-11-03: $25.60 → $26.28 (+2.66%, adj: +1.75%)
  ✓ WBD 2022-02-24: $27.73 → $28.11 (+1.37%, adj: +0.98%)
  ✓ WBD 2022-04-26: $19.83 → $18.15 (-8.47%, adj: -7.49%)
  ✓ WBD 2022-08-04: $17.48 → $13.10 (-

ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found
ERROR:yfinance:$WRK: possibly delisted; no timezone found


  ⚠ WRK 2021-01-28: not enough price data
  ⚠ WRK 2021-05-05: not enough price data
  ⚠ WRK 2021-08-05: not enough price data
  ⚠ WRK 2021-11-09: not enough price data
  ⚠ WRK 2022-02-03: not enough price data
  ⚠ WRK 2022-05-05: not enough price data
  ⚠ WRK 2022-08-04: not enough price data
  ⚠ WRK 2022-11-10: not enough price data
  ⚠ WRK 2023-02-01: not enough price data
  ⚠ WRK 2023-05-04: not enough price data
  ⚠ WRK 2023-08-03: not enough price data
  ✓ WSM 2021-05-26: $77.89 → $77.74 (-0.20%, adj: -0.34%)
  ✓ WSM 2021-08-25: $77.93 → $84.82 (+8.83%, adj: +8.09%)
  ✓ WSM 2021-11-18: $100.29 → $96.90 (-3.38%, adj: -3.05%)
  ✓ WSM 2022-03-16: $70.19 → $73.03 (+4.05%, adj: +1.72%)
  ✓ WSM 2022-05-25: $53.22 → $59.21 (+11.25%, adj: +7.34%)
  ✓ WSM 2022-08-24: $75.62 → $72.20 (-4.53%, adj: -1.86%)
  ✓ WSM 2022-11-17: $61.11 → $58.04 (-5.03%, adj: -6.47%)
  ✓ WSM 2023-03-16: $56.57 → $58.12 (+2.73%, adj: +1.63%)
  ✓ WSM 2023-05-23: $53.15 → $53.98 (+1.57%, adj: +0.14%)
  ✓ WSM 2023-1

ERROR:yfinance:$XLNX: possibly delisted; no timezone found
ERROR:yfinance:$XLNX: possibly delisted; no timezone found


  ⚠ XLNX 2020-07-31: not enough price data
  ⚠ XLNX 2020-10-22: not enough price data
  ✓ XOM 2021-04-30: $47.26 → $50.34 (+6.52%, adj: +6.89%)
  ✓ XOM 2021-07-30: $48.23 → $47.62 (-1.27%, adj: -1.38%)
  ✓ XOM 2021-10-29: $54.82 → $54.37 (-0.84%, adj: -2.03%)
  ✓ XOM 2022-02-01: $69.66 → $70.16 (+0.72%, adj: +1.66%)
  ✓ XOM 2022-04-29: $74.28 → $79.91 (+7.57%, adj: +3.43%)
  ✓ XOM 2022-07-29: $85.33 → $80.13 (-6.10%, adj: -6.69%)
  ✓ XOM 2022-10-28: $98.40 → $97.43 (-0.98%, adj: +2.65%)
  ✓ XOM 2023-01-31: $103.95 → $100.28 (-3.53%, adj: -4.97%)
  ✓ XOM 2023-04-28: $106.85 → $97.45 (-8.80%, adj: -6.89%)
  ✓ XOM 2023-07-28: $94.86 → $95.89 (+1.08%, adj: +2.57%)
  ✓ XOM 2023-10-27: $96.92 → $97.00 (+0.09%, adj: -2.83%)
  ✓ XOM 2024-02-02: $94.49 → $94.72 (+0.25%, adj: -0.51%)
  ✓ XOM 2024-04-26: $110.32 → $108.51 (-1.64%, adj: -0.08%)
  ✓ XOM 2024-08-02: $110.20 → $109.06 (-1.03%, adj: +1.65%)
  ✓ XOM 2024-11-01: $109.25 → $115.00 (+5.26%, adj: +1.76%)
  ✓ XOM 2025-01-31: $102.37 → $105.

In [ ]:
import os
base = '/content/drive/MyDrive/MIS_584_Project'
print("stock_prices.csv:", os.path.exists(f'{base}/stock_prices.csv'))
print("Transcripts_JSON exists:", os.path.isdir(f'{base}/Transcripts_JSON'))
json_count = len([f for f in os.listdir(f'{base}/Transcripts_JSON') if f.endswith('.json')]) if os.path.isdir(f'{base}/Transcripts_JSON') else 0
print(f"JSON files: {json_count}")

stock_prices.csv: True
Transcripts_JSON exists: True
JSON files: 8247
